# 🎵 Melody Wings — Grooming NLI Classifier Fine-tuning

This notebook fine-tunes `distilbert-base-uncased-mnli` on an 18K grooming-specific NLI dataset.

**Steps:**
1. Install deps
2. Upload or clone the dataset
3. Fine-tune for 5 epochs on GPU (~15 min on T4)
4. Download the model weights

**Runtime:** Go to `Runtime > Change runtime type > T4 GPU`

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers[torch] datasets accelerate safetensors

In [ ]:
# Cell 2: Clone your repo and get the dataset
!git clone https://github.com/umangjzx/transcript-Analysis.git --branch fix/celery-mongo-persistence --depth 1
%cd transcript-Analysis/backend
!ls data/

In [ ]:
# Cell 3: Verify GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Load and inspect the dataset
import json
from collections import Counter

with open('data/grooming_nli_dataset.json', 'r') as f:
    samples = json.load(f)

labels = Counter(s['label'] for s in samples)
print(f"Total samples: {len(samples)}")
print(f"Label distribution:")
print(f"  0 (entailment):    {labels[0]} ({labels[0]/len(samples)*100:.1f}%)")
print(f"  1 (neutral):       {labels[1]} ({labels[1]/len(samples)*100:.1f}%)")
print(f"  2 (contradiction): {labels[2]} ({labels[2]/len(samples)*100:.1f}%)")

# Show a few examples
print("\nSample entries:")
for s in samples[:3]:
    print(f"  [{s['label']}] {s['premise'][:60]}... -> {s['hypothesis'][:40]}")

In [ ]:
# Cell 5: Fine-tune the model (5 epochs, ~15 min on T4 GPU)
!python finetune_model.py --epochs 5 --lr 3e-5 --batch-size 32

In [ ]:
# Cell 6: Check training results
import json

metrics_path = 'models/grooming-nli-finetuned/training_metrics.json'
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

print("Training Results")
print("=" * 50)
print(f"Training loss: {metrics['train_loss']:.4f}")
print(f"Epochs: {metrics['epochs']}")
print(f"Learning rate: {metrics['learning_rate']}")
print(f"Train samples: {metrics['train_samples']}")
print(f"Eval samples: {metrics['eval_samples']}")
print(f"\nEval Metrics:")
for k, v in metrics['eval_metrics'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

In [ ]:
# Cell 7: Quick test - verify the model works
from transformers import pipeline

classifier = pipeline(
    'zero-shot-classification',
    model='models/grooming-nli-finetuned',
    device=0
)

test_cases = [
    "Don't tell your mom about our chats, okay?",
    "We should meet at my place when your parents are at work.",
    "Good morning class, please open your textbooks to page 45.",
    "Send me a photo of yourself, I want to see you.",
    "You're so mature for your age, not like other kids.",
]

labels = [
    'asking someone to keep a secret or hide something',
    'arranging an in-person meeting',
    'a safe or normal conversation',
    'requesting a video call or photos',
    'building emotional trust with someone',
    'manipulating or pressuring someone',
    'sexually explicit or inappropriate content',
]

print("Model Predictions")
print("=" * 60)
for text in test_cases:
    result = classifier(text, labels, multi_label=False)
    top = result['labels'][0]
    conf = result['scores'][0]
    print(f"\n  \"{text[:55]}...\"")
    print(f"  -> {top} ({conf:.2%})")

In [ ]:
# Cell 8: Zip and download the model
import shutil

# Remove checkpoints to reduce size
!rm -rf models/grooming-nli-finetuned/checkpoint-*

# Zip the model
shutil.make_archive('grooming-nli-finetuned', 'zip', 'models/grooming-nli-finetuned')

# Get file size
import os
size_mb = os.path.getsize('grooming-nli-finetuned.zip') / (1024*1024)
print(f"Model zip size: {size_mb:.0f} MB")
print("\nDownloading...")

# Auto-download in Colab
from google.colab import files
files.download('grooming-nli-finetuned.zip')

## After downloading:

1. Unzip `grooming-nli-finetuned.zip` into your backend:
   ```
   backend/models/grooming-nli-finetuned/
   ```

2. Set env vars on Cloud Run:
   ```bash
   gcloud run services update audio-safety-backend \
     --region us-central1 --project melodywings \
     --update-env-vars "ENABLE_ML_CLASSIFIER=true,FINETUNED_MODEL_PATH=models/grooming-nli-finetuned"
   ```

3. Redeploy:
   ```bash
   cd New-Rmsi-Latest/backend
   gcloud run deploy audio-safety-backend --source . --region us-central1 \
     --memory 2Gi --cpu 2 --timeout 900 --port 8000 --min-instances=1
   ```

The Dockerfile already has `FINETUNED_MODEL_PATH=models/grooming-nli-finetuned` as a default,
and the model weights will be baked into the image via `COPY . .`